# 재고관리 페이지

**요구사항**: 연차별 FMS/PMS/SMS 잔여비율 확인  
**목적**: 차량 출고 후 연차별 쿠폰(FMS/PMS/SMS) 소진 패턴 분석 및 CR 예측

### 로드 대상 테이블
| DB | 테이블 | 용도 |
|---|---|---|
| Agora (TMKR_L) | l_svc_propo | 서비스 접수 이력 (FMS 유형, 주행거리) |
| Agora (TMKR_L) | l_svc_propo_part | FMS 항목별 사용 이력 |
| Agora (TMKR_L) | l_co_vehic | 차량 기본정보 (출고일) |
| Agora (TMKR_L) | l_om_contract | 계약 정보 (실제 출고일) |
| Agora (TMKR_L) | L_SVC_CR | CR 프로그램 정의 (FMS/PMS/SMS 마스터) |
| Agora (TMKR_L) | l_svc_cr_vehic | 차량별 CR 실행 이력 (ODO 포함) |
| BP/KTWS (KPI_L) | l_svc_propo | 서비스 접수 이력 |
| BP/KTWS (KPI_L) | l_co_vehic | 차량 기본정보 |
| BP/KTWS (KPI_L) | l_om_contract | 계약 정보 |
| Karete | FCT_SERVICE_REPAIR | 정비/수리 팩트 (FMS 포함여부, 금액) |
| Karete | SVC_PROPO | 서비스 접수 (FMS 유형, 주행거리) |
| Karete | SVC_PROPO_PART | FMS 항목별 사용 이력 |
| Karete | CO_VEHIC | 차량 기본정보 (출고일) |
| Karete | OM_CONTRACT | 계약 정보 (실제 출고일) |

## 0. 공통 설정

In [30]:
import os
import pyodbc
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)

# 로컬 저장 경로
DATA_DIR = os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', 'data', 'hnd_svc')
os.makedirs(DATA_DIR, exist_ok=True)
print(f"데이터 저장 경로: {os.path.abspath(DATA_DIR)}")

# 서버 주소
AGORA_SERVER  = "6orm62c43rguff2mpdxwqy76tu-3meqk2mii2pehfjxfscngx5a6e.datawarehouse.fabric.microsoft.com"
BPKTWS_SERVER = "6orm62c43rguff2mpdxwqy76tu-xhehnpiu2bautgglhximxpyqca.datawarehouse.fabric.microsoft.com"
KARETE_SERVER = "6orm62c43rguff2mpdxwqy76tu-kp6lscr4iunung5p66mov6kcka.datawarehouse.fabric.microsoft.com"

def make_conn(server, database=None):
    db_part = f"DATABASE={database};" if database else ""
    conn_str = f"""
    DRIVER={{ODBC Driver 17 for SQL Server}};
    SERVER={server},1433;
    {db_part}
    Encrypt=yes;
    TrustServerCertificate=no;
    Authentication=ActiveDirectoryInteractive;
    Connection Timeout=30;
    """
    return pyodbc.connect(conn_str)

def load_or_fetch(conn, query, filename, label="", force_refresh=False):
    """로컬 parquet 파일이 있으면 로드, 없으면 DB에서 가져와 저장."""
    filepath = os.path.join(DATA_DIR, filename)
    if not force_refresh and os.path.exists(filepath):
        df = pd.read_parquet(filepath)
        print(f"[{label}] 로컬 로드 ← {filename}  ({df.shape[0]:,} rows × {df.shape[1]} cols)")
    else:
        print(f"[{label}] DB 조회 중...")
        df = pd.read_sql(query, conn)
        df.to_parquet(filepath, index=False)
        print(f"[{label}] DB 로드 후 저장 → {filename}  ({df.shape[0]:,} rows × {df.shape[1]} cols)")
    return df

데이터 저장 경로: c:\Project\toyota_project\MCB-ML-toyota_project\louis\data\hnd_svc


---
## 1. Agora (TMKR_L)

In [11]:
conn_agora = make_conn(AGORA_SERVER, "TMKR_L")
pd.read_sql("SELECT DB_NAME() AS current_db", conn_agora)

C:\Users\LouisLee(이시호)\AppData\Local\Temp\ipykernel_3020\1947002203.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql("SELECT DB_NAME() AS current_db", conn_agora)


,current_db
0,TMKR_L


### 1-1. l_svc_propo — 서비스 접수 이력 (FMS 유형 + 주행거리)

In [12]:
agora_svc_propo = load_or_fetch(conn_agora, """
SELECT
    shop_cd,
    propo_dt,
    propo_seq,
    vin,
    svc_model_cd,
    vehic_base_odometer,
    odometer,
    prev_odometer,
    svc_type_cd,
    svc_type_fms_cd,
    stat_cd,
    work_close_yn
FROM dbo.l_svc_propo
WHERE svc_type_fms_cd IS NOT NULL
ORDER BY propo_dt DESC
""", filename="agora_svc_propo.parquet", label="agora_svc_propo")

agora_svc_propo.head()

C:\Users\LouisLee(이시호)\AppData\Local\Temp\ipykernel_3020\1183330764.py:39: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


[agora_svc_propo] DB 조회 중...
[agora_svc_propo] DB 로드 후 저장 → agora_svc_propo.parquet  (715,582 rows × 12 cols)


,shop_cd,propo_dt,propo_seq,vin,svc_model_cd,vehic_base_odometer,odometer,prev_odometer,svc_type_cd,svc_type_fms_cd,stat_cd,work_close_yn
0,YM30104,20250922,184.0,JTHBW1GG2J2177573,AVV60L,0,158040,NaN,20,10000023,60,N
1,YM30101,20250922,558.0,JTHG5LGF5L5008080,VXFA55L,0,100161,NaN,20,10000010,80,N
2,YM30101,20250922,555.0,JTHKD5BH1J2333700,ZWA10,0,92179,NaN,20,10000010,80,N
3,YM30101,20250922,553.0,JTJCJAGA2S2032998,AALH16L,0,368,NaN,20,10000001,80,N
4,YM30101,20250922,535.0,JTJCJAJA6S2059131,AALH15L,0,355,NaN,20,10000001,80,N


### 1-2. l_svc_propo_part — FMS 항목별 사용 이력

In [13]:
agora_svc_propo_part = load_or_fetch(conn_agora, """
SELECT
    shop_cd,
    propo_dt,
    propo_seq,
    part_no,
    seq,
    ro_type_cd,
    settle_type_cd,
    fms_item_cd
FROM dbo.l_svc_propo_part
WHERE fms_item_cd IS NOT NULL
ORDER BY propo_dt DESC
""", filename="agora_svc_propo_part.parquet", label="agora_svc_propo_part")

agora_svc_propo_part.head()

[agora_svc_propo_part] DB 조회 중...


C:\Users\LouisLee(이시호)\AppData\Local\Temp\ipykernel_3020\1183330764.py:39: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


[agora_svc_propo_part] DB 로드 후 저장 → agora_svc_propo_part.parquet  (2,550,629 rows × 8 cols)


,shop_cd,propo_dt,propo_seq,part_no,seq,ro_type_cd,settle_type_cd,fms_item_cd
0,NY30101,20250922,313.0,9043012031,2.0,F,FL,142
1,CW30101,20250922,856.0,9043012031,2.0,F,FL,142
2,SY30101,20250922,503.0,9043012031,2.0,F,FL,141
3,NY30101,20250922,317.0,9043012031,2.0,F,FL,141
4,NY30101,20250922,301.0,9043012031,2.0,F,FL,141


### 1-3. l_co_vehic — 차량 기본정보 (출고일 / 차종)

In [14]:
agora_co_vehic = load_or_fetch(conn_agora, """
SELECT
    vin,
    model_year,
    model_cd,
    svc_model_cd,
    variant_nm,
    delivery_dt,
    odometer,
    base_odometer,
    sales_type,
    first_owner_yn
FROM dbo.l_co_vehic
WHERE delivery_dt IS NOT NULL
""", filename="agora_co_vehic.parquet", label="agora_co_vehic")

agora_co_vehic.head()

[agora_co_vehic] DB 조회 중...


C:\Users\LouisLee(이시호)\AppData\Local\Temp\ipykernel_3020\1183330764.py:39: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


[agora_co_vehic] DB 로드 후 저장 → agora_co_vehic.parquet  (227,613 rows × 10 cols)


,vin,model_year,model_cd,svc_model_cd,variant_nm,delivery_dt,odometer,base_odometer,sales_type,first_owner_yn
0,JTHBW1GG3H2132362,2017,ES,AVV60L,ES300h,2016-08-31,233470,0,NaN,Y
1,JTHBW1GG4G2119795,2016,ES,AVV60L,ES300h,2016-02-18,179120,0,NaN,Y
2,JTHBW1GG1H2154232,2017,ES,AVV60L,ES300h,2017-04-21,151898,0,NaN,Y
3,JTHBW1GG4H2161370,2017,ES,AVV60L,ES300h,2017-08-07,111053,0,NaN,Y
4,JTHBW1GG9G2111711,2016,ES,AVV60L,ES300h,2015-11-20,198315,0,NaN,Y


### 1-4. l_om_contract — 계약 정보 (실제 출고일)

In [15]:
agora_om_contract = load_or_fetch(conn_agora, """
SELECT
    contract_no,
    vin,
    model_cd,
    my_cd,
    delivery_actual_dt,
    last_retail_sales_dt,
    sales_type,
    contract_stat_cd
FROM dbo.l_om_contract
WHERE delivery_actual_dt IS NOT NULL
ORDER BY delivery_actual_dt DESC
""", filename="agora_om_contract.parquet", label="agora_om_contract")

agora_om_contract.head()

[agora_om_contract] DB 조회 중...


C:\Users\LouisLee(이시호)\AppData\Local\Temp\ipykernel_3020\1183330764.py:39: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


[agora_om_contract] DB 로드 후 저장 → agora_om_contract.parquet  (231,924 rows × 8 cols)


,contract_no,vin,model_cd,my_cd,delivery_actual_dt,last_retail_sales_dt,sales_type,contract_stat_cd
0,207305.0,JTJDJRDZ8L5002658,NX,2020,5020-02-25,2020-02-25 13:50:35,11,D4
1,232669.0,JTHBA1C13N2010421,ES,2022,2222-12-12,2022-03-30 12:09:16,96,D4
2,232514.0,NaN,ES,2022,2222-11-11,NaN,11,MZ
3,257685.0,JTHBA1C19P2019384,ES,2023,2031-08-31,2023-08-31 14:54:58,11,D4
4,303178.0,JTJCMAHA9S2017744,RX,2025,2030-11-24,2025-11-27 13:25:04,11,D4


### 1-5. L_SVC_CR — CR 프로그램 정의 마스터 (FMS / PMS / SMS 구분)

In [16]:
agora_l_svc_cr = load_or_fetch(conn_agora, """
SELECT *
FROM dbo.L_SVC_CR
""", filename="agora_l_svc_cr.parquet", label="agora_L_SVC_CR")

agora_l_svc_cr.head()

[agora_L_SVC_CR] DB 조회 중...


C:\Users\LouisLee(이시호)\AppData\Local\Temp\ipykernel_3020\1183330764.py:39: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


[agora_L_SVC_CR] DB 로드 후 저장 → agora_l_svc_cr.parquet  (165 rows × 41 cols)


,CR_NO,CR_TYPE,CR_NM,END_TYPE_CD,APPLY_ST_DATE,APPLY_END_DATE,CAUSE_FRM_NO,CAUSE_PART_NO,CAUSE_PART_COMP_NM,T1_CD,T2_TYPE_CD,T2_CD,TMKR_EXEC_YN,CONDITION,CAUSE,REMEDY,SUBLET_EXPL,GOAL_CNT,RCPT_CNT,STAT_CD,REMARK,USE_YN,TMC_FILE_PATH,TMC_FILE_NM,TMKR_FILE_PATH,TMKR_FILE_NM,REG_DT,REG_USER_ID,UPD_DT,UPD_USER_ID,CR_SHORT_NM,TI_ISSU_DATE,TI_TRF_NO,TI_TITLE,TI_REMEDY,TI_NOTICE,SUB_TYPE,NOTICE_MEMO,PRV_APPLY_ST_DATE,PRV_APPLY_END_DATE,ETL_DT
0,ZKCT-290,R,VGRS Off centered on LEXUS LS,None,20100621,99991231,0514E1A,8918150020,None,99,1,99,NaN,Steering wheel temporarily off centered,TMC campaign ref No ZKCT 290 Jun 2010,Repair per CI ZKCT 290 instruction,None,427.0,454,None,NaN,Y,NaN,NaN,NaN,NaN,2010-06-18 01:44:54,tmr091026,2017-06-30 08:15:42,TMR040865,NaN,20100621,TMKR_SVC_100621_01,VGRS ECU 교환,교체,NaN,R,NaN,NaN,NaN,2024-09-13T14:32:08.4543727Z
1,ZKCT-290-2,R,VGRS Off centered on LEXUS LS,None,20100621,99991231,0516E1A,8918150020,None,99,1,99,NaN,Steering wheel temporarily off centered,TMC campaign ref No ZKCT 290 Jun 2010,Repair per CI ZKCT 290 instruction,None,138.0,157,None,NaN,Y,NaN,NaN,NaN,NaN,2010-06-18 02:06:02,tmr091026,2017-06-30 08:15:36,TMR040865,NaN,20100621,TMKR_SVC_100621_01,VGRS ECU 교환,교체,NaN,R,NaN,NaN,NaN,2024-09-13T14:32:08.4543727Z
2,ZKBA-053-3,R,GR/UR 엔진밸브스프링 시정조치-차량운송관련비용청구,None,20000127,99991231,0528F1A,0400037531,None,99,1,99,NaN,Valve spring degrade may cause spring broken,TMC campaign Ref No ZKBA 053 Jul 2010,Trans or travel expenses for valve spring camp...,None,1.0,1,None,NaN,Y,NaN,NaN,NaN,NaN,2010-07-26 01:41:53,tmr091026,2010-07-26 01:41:53,tmr091026,NaN,NaN,NaN,NaN,NaN,NaN,R,NaN,NaN,NaN,2024-09-13T14:32:08.4543727Z
3,ZKBA-053-5,R,해외반입 GR 엔진밸브스프링 시정조치,None,20101115,99991231,0528F1B,0400037238,None,99,1,99,NaN,Valve spring degrade may cause spring broken,Recall Ref No ZKBA 053 Jul 2010 for gray imported,Repair per CI Ref ZKBA 053 instruction,None,2.0,22,None,NaN,Y,NaN,NaN,NaN,NaN,2010-11-10 23:52:05,TMR050877,2017-09-21 00:07:44,CT13005,NaN,NaN,NaN,NaN,NaN,NaN,Y,NaN,NaN,NaN,2024-09-13T14:32:08.4543727Z
4,ZKBA-084-3,R,브레이크 마스터 실린더 시정조치-이전 유상수리비 환불,None,20101220,99991231,0516K6,0400033158,None,99,1,99,NaN,Brake oil leak from rear brake master seal,TMC campaign ref No ZKBA 084 Nov 2010,Refund previously paid by customer,None,NaN,12,None,딜러에서는 고객에게 이전 유상수리비를 환불해 드렸음을 입증할 수 있는 서류를 보관 ...,Y,NaN,NaN,NaN,NaN,2010-12-15 02:18:03,tmr091026,2011-06-14 01:51:51,NY030709,NaN,NaN,NaN,NaN,NaN,NaN,R,NaN,NaN,NaN,2024-09-13T14:32:08.4543727Z


### 1-6. l_svc_cr_vehic — 차량별 CR 실행 이력 (ODO + 실행일)

In [17]:
agora_svc_cr_vehic = load_or_fetch(conn_agora, """
SELECT
    cr_no,
    vin,
    exec_yn,
    exec_date,
    propo_shop_cd,
    propo_dt,
    propo_seq,
    repair_dt,
    odo,
    cr_case_no
FROM dbo.l_svc_cr_vehic
ORDER BY propo_dt DESC
""", filename="agora_svc_cr_vehic.parquet", label="agora_svc_cr_vehic")

agora_svc_cr_vehic.head()

[agora_svc_cr_vehic] DB 조회 중...


C:\Users\LouisLee(이시호)\AppData\Local\Temp\ipykernel_3020\1183330764.py:39: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


[agora_svc_cr_vehic] DB 로드 후 저장 → agora_svc_cr_vehic.parquet  (264,563 rows × 10 cols)


,cr_no,vin,exec_yn,exec_date,propo_shop_cd,propo_dt,propo_seq,repair_dt,odo,cr_case_no
0,25SD 108,JTHBA1C17T2029230,Y,20260623,DT30106,20260623,587.0,NaN,NaN,NaN
1,25SD 108,JTHBA1C15P2017616,Y,20260623,JB30102,20260623,374.0,NaN,NaN,NaN
2,25SD 108,JTHBA1C14S2025912,Y,20260623,PM30102,20260623,512.0,NaN,NaN,NaN
3,25SD 108,JTHBA1C12S2025567,Y,20260623,DT30101,20260623,898.0,NaN,NaN,NaN
4,25SD 019,JTJHKAEZ2R2019920,Y,20260623,DT30108,20260623,449.0,NaN,NaN,NaN


---
## 2. BP/KTWS (KPI_L)

In [18]:
conn_bpktws = make_conn(BPKTWS_SERVER, "KPI_L")
pd.read_sql("SELECT DB_NAME() AS current_db", conn_bpktws)

C:\Users\LouisLee(이시호)\AppData\Local\Temp\ipykernel_3020\1065623051.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql("SELECT DB_NAME() AS current_db", conn_bpktws)


,current_db
0,KPI_L


### 2-1. l_svc_propo — 서비스 접수 이력 (FMS 유형 + 주행거리)

In [21]:
bpktws_svc_propo = load_or_fetch(conn_bpktws, """
SELECT
    SHOP_CD,
    PROPO_DT,
    PROPO_SEQ,
    VIN,
    SVC_MODEL_CD,
    VEHIC_BASE_ODOMETER,
    ODOMETER,
    PREV_ODOMETER,
    SVC_TYPE_CD,
    SVC_TYPE_FMS_CD,
    STAT_CD,
    WORK_CLOSE_YN
FROM dbo.l_svc_propo
WHERE SVC_TYPE_FMS_CD IS NOT NULL
ORDER BY PROPO_DT DESC
""", filename="bpktws_svc_propo.parquet", label="bpktws_svc_propo")

bpktws_svc_propo.head()

[bpktws_svc_propo] DB 조회 중...


C:\Users\LouisLee(이시호)\AppData\Local\Temp\ipykernel_3020\1183330764.py:39: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


[bpktws_svc_propo] DB 로드 후 저장 → bpktws_svc_propo.parquet  (775,020 rows × 12 cols)


,SHOP_CD,PROPO_DT,PROPO_SEQ,VIN,SVC_MODEL_CD,VEHIC_BASE_ODOMETER,ODOMETER,PREV_ODOMETER,SVC_TYPE_CD,SVC_TYPE_FMS_CD,STAT_CD,WORK_CLOSE_YN
0,CW30104,20260623,112,JTHBA1C15T2031395,AXZH11L,0,3033,NaN,20,10000002,60,N
1,YM30106,20260623,82,JTJBGMCA3M2074729,GYL25,0,134047,NaN,20,10000009,60,N
2,CT30103,20260623,598,JTJHKAFZ7S2067594,AAZH26L,0,18653,NaN,20,10000002,65,N
3,PM30102,20260623,522,JTHBW1GG6H2133442,AVV60L,0,141198,NaN,20,10000022,60,N
4,PM30101,20260623,714,JTJGD7CX8S4007534,VJH310L,0,9942,NaN,20,10000002,80,N


### 2-2. l_co_vehic — 차량 기본정보 (출고일 / 차종)

In [22]:
bpktws_co_vehic = load_or_fetch(conn_bpktws, """
SELECT
    vin,
    model_year,
    model_cd,
    svc_model_cd,
    variant_nm,
    delivery_dt,
    odometer,
    base_odometer,
    sales_type,
    first_owner_yn
FROM dbo.l_co_vehic
WHERE delivery_dt IS NOT NULL
""", filename="bpktws_co_vehic.parquet", label="bpktws_co_vehic")

bpktws_co_vehic.head()

[bpktws_co_vehic] DB 조회 중...


C:\Users\LouisLee(이시호)\AppData\Local\Temp\ipykernel_3020\1183330764.py:39: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


[bpktws_co_vehic] DB 로드 후 저장 → bpktws_co_vehic.parquet  (227,613 rows × 10 cols)


,vin,model_year,model_cd,svc_model_cd,variant_nm,delivery_dt,odometer,base_odometer,sales_type,first_owner_yn
0,JTHBW1GG3H2132362,2017,ES,AVV60L,ES300h,2016-08-31,233470,0,NaN,Y
1,JTHBW1GG4G2119795,2016,ES,AVV60L,ES300h,2016-02-18,179120,0,NaN,Y
2,JTHBW1GG1H2154232,2017,ES,AVV60L,ES300h,2017-04-21,151898,0,NaN,Y
3,JTHBW1GG4H2161370,2017,ES,AVV60L,ES300h,2017-08-07,111053,0,NaN,Y
4,JTHBW1GG9G2111711,2016,ES,AVV60L,ES300h,2015-11-20,198315,0,NaN,Y


### 2-3. l_om_contract — 계약 정보 (실제 출고일)

In [23]:
bpktws_om_contract = load_or_fetch(conn_bpktws, """
SELECT
    contract_no,
    vin,
    model_cd,
    my_cd,
    delivery_actual_dt,
    last_retail_sales_dt,
    sales_type,
    contract_stat_cd
FROM dbo.l_om_contract
WHERE delivery_actual_dt IS NOT NULL
ORDER BY delivery_actual_dt DESC
""", filename="bpktws_om_contract.parquet", label="bpktws_om_contract")

bpktws_om_contract.head()

[bpktws_om_contract] DB 조회 중...


C:\Users\LouisLee(이시호)\AppData\Local\Temp\ipykernel_3020\1183330764.py:39: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


[bpktws_om_contract] DB 로드 후 저장 → bpktws_om_contract.parquet  (231,924 rows × 8 cols)


,contract_no,vin,model_cd,my_cd,delivery_actual_dt,last_retail_sales_dt,sales_type,contract_stat_cd
0,207305.0,JTJDJRDZ8L5002658,NX,2020,5020-02-25,2020-02-25 13:50:35,11,D4
1,232669.0,JTHBA1C13N2010421,ES,2022,2222-12-12,2022-03-30 12:09:16,96,D4
2,232514.0,NaN,ES,2022,2222-11-11,NaN,11,MZ
3,257685.0,JTHBA1C19P2019384,ES,2023,2031-08-31,2023-08-31 14:54:58,11,D4
4,303178.0,JTJCMAHA9S2017744,RX,2025,2030-11-24,2025-11-27 13:25:04,11,D4


---
## 3. Karete

In [31]:
# Karete 서버 DB 목록 확인
conn_karete_base = make_conn(KARETE_SERVER)
karete_dbs = pd.read_sql("SELECT name FROM sys.databases ORDER BY name", conn_karete_base)
conn_karete_base.close()

SYSTEM_DBS = {'master', 'tempdb', 'model', 'msdb'}
karete_user_dbs = karete_dbs[~karete_dbs['name'].isin(SYSTEM_DBS)]['name'].tolist()
print("Karete 사용자 DB 목록:", karete_user_dbs)

# 각 DB에서 특정 테이블의 존재 여부 탐색
def find_db_for_table(server, db_list, table_name, schema='dbo'):
    for db in db_list:
        try:
            conn = make_conn(server, db)
            result = pd.read_sql(
                f"SELECT COUNT(*) AS cnt FROM INFORMATION_SCHEMA.TABLES "
                f"WHERE TABLE_SCHEMA='{schema}' AND TABLE_NAME='{table_name}'",
                conn
            )
            conn.close()
            if result['cnt'].iloc[0] > 0:
                return db
        except Exception as e:
            print(f"  [{db}] 접근 실패: {e}")
    return None

print("\n각 테이블의 DB 탐색 중...")
KARETE_FCT_DB = find_db_for_table(KARETE_SERVER, karete_user_dbs, 'FCT_SERVICE_REPAIR')
KARETE_SVC_DB = find_db_for_table(KARETE_SERVER, karete_user_dbs, 'SVC_PROPO')

print(f"\nFCT_SERVICE_REPAIR → {KARETE_FCT_DB}")
print(f"SVC_PROPO          → {KARETE_SVC_DB}")

conn_karete_fct = make_conn(KARETE_SERVER, KARETE_FCT_DB)
conn_karete_svc = make_conn(KARETE_SERVER, KARETE_SVC_DB)

C:\Users\LouisLee(이시호)\AppData\Local\Temp\ipykernel_3020\3364521199.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  karete_dbs = pd.read_sql("SELECT name FROM sys.databases ORDER BY name", conn_karete_base)
C:\Users\LouisLee(이시호)\AppData\Local\Temp\ipykernel_3020\3364521199.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql(


Karete 사용자 DB 목록: ['LH_INTELLIGENCE_BI', 'LH_INTELLIGENCE_ML', 'LH_META', 'LH_REFINED', 'LH_STAGING']

각 테이블의 DB 탐색 중...


C:\Users\LouisLee(이시호)\AppData\Local\Temp\ipykernel_3020\3364521199.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql(
C:\Users\LouisLee(이시호)\AppData\Local\Temp\ipykernel_3020\3364521199.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql(
C:\Users\LouisLee(이시호)\AppData\Local\Temp\ipykernel_3020\3364521199.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql(
C:\Users\LouisLee(이시호)\AppData\Local\Temp\ipykernel_3020\3364521199.py:15: UserWarning: pandas only 


FCT_SERVICE_REPAIR → LH_INTELLIGENCE_BI
SVC_PROPO          → LH_REFINED


In [32]:
print(f"FCT_SERVICE_REPAIR → {KARETE_FCT_DB}  (conn_karete_fct)")
print(f"SVC_PROPO          → {KARETE_SVC_DB}  (conn_karete_svc)")
print(f"SVC_PROPO_PART     → {KARETE_SVC_DB}  (conn_karete_svc)")
print(f"CO_VEHIC           → {KARETE_SVC_DB}  (conn_karete_svc)")
print(f"OM_CONTRACT        → {KARETE_SVC_DB}  (conn_karete_svc)")

FCT_SERVICE_REPAIR → LH_INTELLIGENCE_BI  (conn_karete_fct)
SVC_PROPO          → LH_REFINED  (conn_karete_svc)
SVC_PROPO_PART     → LH_REFINED  (conn_karete_svc)
CO_VEHIC           → LH_REFINED  (conn_karete_svc)
OM_CONTRACT        → LH_REFINED  (conn_karete_svc)


### 3-1. FCT_SERVICE_REPAIR — 정비/수리 팩트 (FMS 포함여부 + 금액)

In [ ]:
karete_fct_svc_repair = load_or_fetch(conn_karete_fct, """
SELECT
    SOURCE_SYSTEM,
    SHOP_CODE,
    SHOP_NAME,
    RO_DATE,
    RO_SEQUENCE,
    VIN,
    VEHICLE_MODEL_YEAR,
    VEHICLE_MODEL,
    VEHICLE_SERVICE_MODEL_CODE,
    RO_ODOMETER,
    RO_STATUS,
    FMS_AMOUNT,
    IS_FMS_INCLUDE,
    WARRANTY_AMOUNT,
    IS_WARRANTY_INCLUDE,
    RO_CLOSE_DATE
FROM dbo.FCT_SERVICE_REPAIR
ORDER BY RO_DATE DESC
""", filename="karete_fct_svc_repair.parquet", label="karete_FCT_SERVICE_REPAIR")

karete_fct_svc_repair.head()

### 3-2. SVC_PROPO — 서비스 접수 (FMS 유형 + 주행거리)

In [33]:
karete_svc_propo = load_or_fetch(conn_karete_svc, """
SELECT
    SOURCE_SYSTEM,
    SHOP_CODE,
    SHOP_NAME,
    PROPO_DATE,
    PROPO_SEQ,
    VIN,
    SERVICE_MODEL_CODE,
    BASE_ODOMETER,
    ODOMETER,
    PREVIOUS_ODOMETER,
    SERVICE_TYPE,
    SERVICE_TYPE_FMS,
    PROPO_STAT_NAME,
    END_GB
FROM dbo.SVC_PROPO
WHERE SERVICE_TYPE_FMS IS NOT NULL
ORDER BY PROPO_DATE DESC
""", filename="karete_svc_propo.parquet", label="karete_SVC_PROPO")

karete_svc_propo.head()

[karete_SVC_PROPO] DB 조회 중...


C:\Users\LouisLee(이시호)\AppData\Local\Temp\ipykernel_3020\1183330764.py:39: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


[karete_SVC_PROPO] DB 로드 후 저장 → karete_svc_propo.parquet  (5,072,378 rows × 14 cols)


,SOURCE_SYSTEM,SHOP_CODE,SHOP_NAME,PROPO_DATE,PROPO_SEQ,VIN,SERVICE_MODEL_CODE,BASE_ODOMETER,ODOMETER,PREVIOUS_ODOMETER,SERVICE_TYPE,SERVICE_TYPE_FMS,PROPO_STAT_NAME,END_GB
0,LDMS,PM30101,렉서스서초서비스센터,20260622,1,JTHBW1GG7J2170439,AVV60L,0.0,153234.0,0,EM 점검,,정산,
1,LDMS,PM30101,렉서스서초서비스센터,20260622,2,JTHBW1GG7J2170439,AVV60L,0.0,153235.0,0,EM 점검,,정산,
2,LDMS,KM30102,렉서스부산서비스센터,20260622,12,JTHKD5BH1J2332160,ZWA10,0.0,157.0,0,일반 정비,,접수,
3,LDMS,KM30102,렉서스부산서비스센터,20260622,14,JTHBA1C19T2029522,AXZH11L,0.0,16676.0,0,EM 점검,,접수,
4,TDMS,DM30101,토요타부산서비스센터,20260622,36,JTDKBRFU3H3559991,ZVW50L,0.0,86015.0,0,일반 정비,,작업중,


### 3-3. SVC_PROPO_PART — FMS 항목별 사용 이력

In [34]:
karete_svc_propo_part = load_or_fetch(conn_karete_svc, """
SELECT
    SOURCE_SYSTEM,
    SHOP_CODE,
    PROPO_DATE,
    PROPO_SEQ,
    PART_NUMBER,
    PART_NAME,
    SEQUENCE,
    SETTLE_TYPE_NAME,
    FMS_ITEM_CODE,
    PSP_CODE,
    PM_CODE,
    REAL_ISSUE_QUANTITY,
    CONFIRM_AMOUNT,
    CANCEL_YN
FROM dbo.SVC_PROPO_PART
WHERE FMS_ITEM_CODE IS NOT NULL
ORDER BY PROPO_DATE DESC
""", filename="karete_svc_propo_part.parquet", label="karete_SVC_PROPO_PART")

karete_svc_propo_part.head()

[karete_SVC_PROPO_PART] DB 조회 중...


C:\Users\LouisLee(이시호)\AppData\Local\Temp\ipykernel_3020\1183330764.py:39: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


[karete_SVC_PROPO_PART] DB 로드 후 저장 → karete_svc_propo_part.parquet  (21,063,781 rows × 14 cols)


,SOURCE_SYSTEM,SHOP_CODE,PROPO_DATE,PROPO_SEQ,PART_NUMBER,PART_NAME,SEQUENCE,SETTLE_TYPE_NAME,FMS_ITEM_CODE,PSP_CODE,PM_CODE,REAL_ISSUE_QUANTITY,CONFIRM_AMOUNT,CANCEL_YN
0,TDMS,DM30101,20260619,31,0411128832,"GASKET KIT, ENGINE",1,리콜,,,,1.0,280700.0,N
1,TDMS,TD30104,20260619,8,0400005133,"PLATE, ACCELERATOR (t 1.4)",1,일반,,,,0.0,300.0,N
2,TDMS,DM30101,20260616,25,0411128832,"GASKET KIT, ENGINE",1,리콜,,,,1.0,280700.0,N
3,TDMS,DM30101,20260616,24,0411128832,"GASKET KIT, ENGINE",1,리콜,,,,1.0,280700.0,N
4,TDMS,TD30101,20260610,6,7539235220,RETAINER OUTSIDE,1,일반,,,,1.0,2500.0,N


### 3-4. CO_VEHIC — 차량 기본정보 (출고일 / 차종)

In [35]:
karete_co_vehic = load_or_fetch(conn_karete_svc, """
SELECT
    SOURCE_SYSTEM,
    VIN,
    VEHICLE_MODEL_YEAR,
    VEHICLE_MODEL_CODE,
    SERVICE_MODEL_CODE,
    VEHICLE_VARIANT_NAME,
    DELIVERY_DATE,
    ODOMETER,
    BASE_ODOMETER,
    SALES_TYPE,
    FIRST_OWNER_YN
FROM dbo.CO_VEHIC
WHERE DELIVERY_DATE IS NOT NULL
""", filename="karete_co_vehic.parquet", label="karete_CO_VEHIC")

karete_co_vehic.head()

[karete_CO_VEHIC] DB 조회 중...


C:\Users\LouisLee(이시호)\AppData\Local\Temp\ipykernel_3020\1183330764.py:39: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


[karete_CO_VEHIC] DB 로드 후 저장 → karete_co_vehic.parquet  (357,365 rows × 11 cols)


,SOURCE_SYSTEM,VIN,VEHICLE_MODEL_YEAR,VEHICLE_MODEL_CODE,SERVICE_MODEL_CODE,VEHICLE_VARIANT_NAME,DELIVERY_DATE,ODOMETER,BASE_ODOMETER,SALES_TYPE,FIRST_OWNER_YN
0,LDMS,JTHBA1C10P2015627,2023,ES,AXZH11L,ES300h(N),2023-06-21,28863.0,0.0,NaN,Y
1,LDMS,JTHBA1C10R2020877,2024,ES,AXZH11L,ES300h(N),2023-09-27,21408.0,0.0,NaN,Y
2,LDMS,JTHBA1C19T2029522,2026,ES,AXZH11L,ES300h(N),2025-01-24,16675.0,0.0,NaN,Y
3,LDMS,JTHBF30G830108195,2003,ES,MCV30,ES300,2002-12-10,206244.0,0.0,11,Y
4,LDMS,JTHBW1GG7J2170439,2018,ES,AVV60L,ES300h,2017-12-29,153235.0,0.0,NaN,Y


### 3-5. OM_CONTRACT — 계약 정보 (실제 출고일)

In [36]:
karete_om_contract = load_or_fetch(conn_karete_svc, """
SELECT
    SOURCE_SYSTEM,
    CONTRACT_NUMBER,
    VIN,
    MODEL_CODE,
    MODEL_NAME,
    MODEL_YEAR,
    DELIVERY_ACTUAL_DATE,
    LAST_RETAIL_SALES_DATE,
    SALES_TYPE_NAME,
    CONTRACT_STAT_NAME
FROM dbo.OM_CONTRACT
WHERE DELIVERY_ACTUAL_DATE IS NOT NULL
ORDER BY DELIVERY_ACTUAL_DATE DESC
""", filename="karete_om_contrapip fweffewfewfwfewfefwfewfewfewfwefewfdfsfwfct.parquet", label="karete_OM_CONTRACT")

karete_om_contract.head()

[karete_OM_CONTRACT] DB 조회 중...


C:\Users\LouisLee(이시호)\AppData\Local\Temp\ipykernel_3020\1183330764.py:39: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


[karete_OM_CONTRACT] DB 로드 후 저장 → karete_om_contract.parquet  (384,691 rows × 10 cols)


,SOURCE_SYSTEM,CONTRACT_NUMBER,VIN,MODEL_CODE,MODEL_NAME,MODEL_YEAR,DELIVERY_ACTUAL_DATE,LAST_RETAIL_SALES_DATE,SALES_TYPE_NAME,CONTRACT_STAT_NAME
0,TDMS,102646,JTNBF3EK1A3004038,CAM,CAMRY,2010,90091106,2009-11-09 10:47:47,노후차량,출고완료
1,TDMS,263083,JF1ZNBB13R9755121,T86,TOYOTA 86,2024,84040229,2024-04-09 09:21:05,일반고객,출고완료
2,TDMS,180103,4T1BD1FK3HU226609,CAM,CAMRY,2017,67050410,2017-07-12 08:57:37,일반고객,출고완료
3,TDMS,171491,JTDKBRFU3H3536811,PRI,PRIUS,2017,61161022,2016-10-24 16:33:12,일반고객,출고완료
4,TDMS,218104,JTMDWRFV7LD044016,RAV,RAV4,2020,60191130,2019-11-05 13:19:05,일반고객,출고완료


---
## 4. 로드 결과 요약

In [37]:
summary = {
    "agora_svc_propo":       agora_svc_propo.shape,
    "agora_svc_propo_part":  agora_svc_propo_part.shape,
    "agora_co_vehic":        agora_co_vehic.shape,
    "agora_om_contract":     agora_om_contract.shape,
    "agora_l_svc_cr":        agora_l_svc_cr.shape,
    "agora_svc_cr_vehic":    agora_svc_cr_vehic.shape,
    "bpktws_svc_propo":      bpktws_svc_propo.shape,
    "bpktws_co_vehic":       bpktws_co_vehic.shape,
    "bpktws_om_contract":    bpktws_om_contract.shape,
    "karete_fct_svc_repair": karete_fct_svc_repair.shape,
    "karete_svc_propo":      karete_svc_propo.shape,
    "karete_svc_propo_part": karete_svc_propo_part.shape,
    "karete_co_vehic":       karete_co_vehic.shape,
    "karete_om_contract":    karete_om_contract.shape,
}

result_df = pd.DataFrame(
    [(k, v[0], v[1]) for k, v in summary.items()],
    columns=["DataFrame", "rows", "cols"]
)

# 저장된 parquet 파일 목록 및 크기
files = [
    (f, os.path.getsize(os.path.join(DATA_DIR, f)) / 1024 / 1024)
    for f in os.listdir(DATA_DIR) if f.endswith('.parquet')
]
file_df = pd.DataFrame(sorted(files), columns=["파일명", "크기(MB)"])
file_df["크기(MB)"] = file_df["크기(MB)"].round(1)

print("=== 로드된 DataFrame ===")
display(result_df)
print(f"\n=== 저장 경로: {os.path.abspath(DATA_DIR)} ===")
display(file_df)

=== 로드된 DataFrame ===


,DataFrame,rows,cols
0,agora_svc_propo,715582,12
1,agora_svc_propo_part,2550629,8
2,agora_co_vehic,227613,10
3,agora_om_contract,231924,8
4,agora_l_svc_cr,165,41
5,agora_svc_cr_vehic,264563,10
6,bpktws_svc_propo,775020,12
7,bpktws_co_vehic,227613,10
8,bpktws_om_contract,231924,8
9,karete_fct_svc_repair,571275,16



=== 저장 경로: c:\Project\toyota_project\MCB-ML-toyota_project\louis\data\hnd_svc ===


,파일명,크기(MB)
0,agora_co_vehic.parquet,3.7
1,agora_l_svc_cr.parquet,0.0
2,agora_om_contract.parquet,4.7
3,agora_svc_cr_vehic.parquet,2.9
4,agora_svc_propo.parquet,11.3
5,agora_svc_propo_part.parquet,11.8
6,bpktws_co_vehic.parquet,3.7
7,bpktws_om_contract.parquet,4.7
8,bpktws_svc_propo.parquet,12.4
9,karete_co_vehic.parquet,5.3
